In [ ]:
# ================================================================
# COMFYQUEUE WORKER — Celula 1: Configuracao
# ================================================================
# Ajuste as variaveis abaixo e execute esta celula primeiro.
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

# 2. Caminhos (Importante: Usar aspas se for usar no shell)
DRIVE_COMFYUI_PATH = "/content/drive/Shared drives/IA/ComfyUI"
COMFYUI_REPO_PATH = "/content/ComfyUI"
COMFYUI_PORT = 8188

LARAVEL_API_URL = "https://seu-portal.com"  # URL base do servidor Laravel (sem barra final)

# Sem API key por enquanto (teste)
HEADERS = {"Content-Type": "application/json"}

print("Configuracao carregada")
print(f"  Servidor Laravel : {LARAVEL_API_URL}")
print(f"  ComfyUI repo     : {COMFYUI_REPO_PATH}")
print(f"  ComfyUI drive    : {DRIVE_COMFYUI_PATH}")
print(f"  Porta ComfyUI    : {COMFYUI_PORT}")

## Celula 2 - Instalar ComfyUI

Clona (ou atualiza) o repositorio e instala as dependencias Python. Tambem prepara links simbolicos para usar os modelos armazenados no Google Drive.

In [ ]:
import os, subprocess, sys
from pathlib import Path

def run(cmd, **kwargs):
    """Executa um comando shell e gera excecao se falhar."""
    subprocess.run(cmd, shell=True, check=True, **kwargs)

# Clonar ou atualizar ComfyUI repo local
if not os.path.exists(COMFYUI_REPO_PATH):
    print("Clonando ComfyUI...")
    run(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFYUI_REPO_PATH}")
else:
    print("Atualizando ComfyUI...")
    run("git pull", cwd=COMFYUI_REPO_PATH)

# Instalar dependencias Python
print("Instalando dependencias...")
run(f"{sys.executable} -m pip install -r {COMFYUI_REPO_PATH}/requirements.txt -q")

# Garantir pastas persistentes no Drive
Path(DRIVE_COMFYUI_PATH).mkdir(parents=True, exist_ok=True)
Path(f"{DRIVE_COMFYUI_PATH}/models").mkdir(parents=True, exist_ok=True)
Path(f"{DRIVE_COMFYUI_PATH}/output").mkdir(parents=True, exist_ok=True)

# Apontar repo local para usar modelos/output no Drive
for folder in ["models", "output"]:
    local_path = Path(COMFYUI_REPO_PATH) / folder
    drive_path = Path(DRIVE_COMFYUI_PATH) / folder

    if local_path.exists() and not local_path.is_symlink():
        # Preserva conteudo original no primeiro setup
        backup = Path(COMFYUI_REPO_PATH) / f"{folder}_backup"
        if not backup.exists():
            local_path.rename(backup)
        else:
            run(f"rm -rf '{local_path}'")

    if local_path.is_symlink() and os.readlink(local_path) != str(drive_path):
        local_path.unlink()

    if not local_path.exists():
        local_path.symlink_to(drive_path, target_is_directory=True)

print("\nComfyUI instalado e configurado para usar modelos no Drive")

## Célula 3 — Baixar modelos

Busca os modelos necessários dos jobs **pendentes** no Laravel e faz download apenas dos que ainda não existem no ComfyUI.

In [ ]:
import requests
from pathlib import Path

def download_models():
    resp = requests.get(f"{LARAVEL_API_URL}/api/comfy-queue/models", headers=HEADERS, timeout=30)
    resp.raise_for_status()

    models = resp.json()
    if not models:
        print("Nenhum modelo necessario para os jobs pendentes.")
        return

    print(f"{len(models)} modelo(s) necessario(s):\n")

    for model in models:
        name = model.get("name", "")
        dest = model.get("dest", "models/checkpoints")
        url  = model.get("url", "")

        if not name or not url:
            print(f"  Entrada invalida (sem nome ou URL): {model}")
            continue

        # Modelos ficam no Drive
        clean_dest = dest.removeprefix("/")
        clean_dest = clean_dest.removeprefix("models/")
        dest_dir  = Path(DRIVE_COMFYUI_PATH) / "models" / clean_dest
        dest_file = dest_dir / name
        dest_dir.mkdir(parents=True, exist_ok=True)

        if dest_file.exists():
            print(f"  {name} - ja existe, pulando")
            continue

        print(f"  Baixando {name}...")
        with requests.get(url, stream=True, timeout=3600) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            downloaded = 0
            with open(dest_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        print(f"    {downloaded / total * 100:.1f}%", end="\r")
        print(f"  {name} - concluido")

    print("\nModelos prontos no Drive")

download_models()

## Célula 4 — Executar jobs

Inicia o ComfyUI, busca os jobs pendentes um a um, envia cada workflow ao ComfyUI e reporta o resultado (sucesso ou erro) de volta ao Laravel.

In [ ]:
import requests, subprocess, sys, time

COMFYUI_BASE = f"http://127.0.0.1:{COMFYUI_PORT}"

# Utilitarios

def start_comfyui():
    """Inicia o processo do ComfyUI e aguarda ficar disponivel (ate 3 min)."""
    print("Iniciando ComfyUI...")
    proc = subprocess.Popen(
        [sys.executable, "main.py", "--port", str(COMFYUI_PORT), "--listen", "--disable-auto-launch"],
        cwd=COMFYUI_REPO_PATH,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for attempt in range(60):
        time.sleep(3)
        try:
            r = requests.get(f"{COMFYUI_BASE}/system_stats", timeout=3)
            if r.status_code == 200:
                print(f"ComfyUI disponivel (tentativa {attempt + 1})")
                return proc
        except requests.exceptions.ConnectionError:
            pass
    proc.terminate()
    raise RuntimeError("ComfyUI nao respondeu em 3 minutos")


def submit_workflow(workflow: dict) -> str:
    """Envia o workflow ao ComfyUI e retorna o prompt_id."""
    r = requests.post(f"{COMFYUI_BASE}/prompt", json={"prompt": workflow}, timeout=30)
    r.raise_for_status()
    return r.json()["prompt_id"]


def wait_for_result(prompt_id: str, timeout_sec: int = 600) -> dict:
    """Aguarda a conclusao do prompt e retorna o historico."""
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        time.sleep(5)
        r = requests.get(f"{COMFYUI_BASE}/history/{prompt_id}", timeout=10)
        if r.status_code == 200:
            history = r.json()
            if prompt_id in history:
                return history[prompt_id]
    raise TimeoutError(f"Job {prompt_id} nao concluido em {timeout_sec}s")


def extract_output_files(history: dict) -> list:
    """Extrai a lista de arquivos gerados do historico do ComfyUI."""
    files = []
    for node_output in history.get("outputs", {}).values():
        for img in node_output.get("images", []):
            files.append({
                "filename": img["filename"],
                "subfolder": img.get("subfolder", ""),
                "type": img.get("type", "output"),
            })
    return files


# Loop principal
comfyui_proc = start_comfyui()
processed = 0
failed = 0

try:
    while True:
        resp = requests.get(f"{LARAVEL_API_URL}/api/comfy-queue/next", headers=HEADERS, timeout=30)

        if resp.status_code == 204:
            print("\nNenhum job pendente. Worker finalizado.")
            break

        resp.raise_for_status()
        job = resp.json()
        job_id = job["id"]
        print(f"\n[Job #{job_id}] tipo: {job['type']}")

        try:
            prompt_id = submit_workflow(job["params"])
            print(f"  prompt_id: {prompt_id}")

            result = wait_for_result(prompt_id)
            output_files = extract_output_files(result)

            requests.post(
                f"{LARAVEL_API_URL}/api/comfy-queue/job/{job_id}/done",
                headers=HEADERS,
                json={"output_files": output_files, "prompt_id": prompt_id},
                timeout=30,
            ).raise_for_status()

            processed += 1
            print(f"  Concluido - {len(output_files)} arquivo(s)")

        except Exception as exc:
            requests.post(
                f"{LARAVEL_API_URL}/api/comfy-queue/job/{job_id}/fail",
                headers=HEADERS,
                json={"error": str(exc)},
                timeout=30,
            ).raise_for_status()
            failed += 1
            print(f"  Falhou: {exc}")

finally:
    comfyui_proc.terminate()
    print(f"\nResumo: {processed} concluido(s), {failed} falha(s)")